# Phase 6: Baseline, Machine Learning Modeling, and Cross-Validation

**Project:** Smart Student Retention & Early Warning System  
**Cutoff Horizon:** Relative Day 35 ($t \le 35$)  
**Target:** `withdrawn_after_cutoff` (~17.4% positive base rate under Policy C)  
**Feature Allowlist:** Strictly locked 22 primary non-demographic features  

---

## 1. Architectural & Evaluation Principles

1. **Student Group Disjointness (`GroupShuffleSplit` & `GroupKFold`):** Learners frequently enroll in multiple course presentations. Random row splitting leaks identical student behavioral patterns between train and test, producing overly optimistic evaluations. We enforce strictly zero student ID overlap across partitions.
2. **Strict Pipeline Encapsulation:** Imputation (median + indicator for numerical; constant for categorical), standard scaling, and one-hot encoding are fitted strictly on the training partition inside each cross-validation fold.
3. **Candidate Baseline Hierarchy:**
   - **Model A (Dummy Baseline):** Predicts positive class prior distribution to prove meaningful learning over naive guessing.
   - **Model B (Logistic Regression):** Interpretable linear baseline with L2 penalty, providing log-odds coefficients and Odds Ratios.
   - **Model C (Random Forest):** Non-linear ensemble baseline (200 trees, depth 8) capturing complex non-linear feature interactions.
4. **Validation-Derived Operational Thresholding:** The decision threshold is selected strictly on validation/cross-validation folds by maximizing F1 subject to operational precision constraint ($\text{precision} \ge 0.35$). The held-out test partition is never used for tuning.

In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "pyproject.toml").is_file():
            return directory
    return current.parent

PROJECT_ROOT = find_project_root()
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features" / "feature_matrix_policy_c.parquet"
print(f"Project root: {PROJECT_ROOT}")
print(f"Feature matrix: {FEATURES_PATH} (exists: {FEATURES_PATH.exists()})")

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from student_retention.features.feature_selection import select_primary_features
from student_retention.modeling.splits import (
    split_by_student_group,
    verify_zero_group_overlap,
)

# Load empirical feature matrix
raw_df = pd.read_parquet(FEATURES_PATH)
clean_df = select_primary_features(raw_df, include_target=True, include_identifiers=True)

# Perform group-aware split (80% train, 20% test)
train_df, test_df = split_by_student_group(clean_df, test_size=0.20, random_state=42, group_col="id_student")
verify_zero_group_overlap(train_df, test_df, group_col="id_student")

print(f"Total cohort records: {len(clean_df):,}")
print(f"Train partition: {len(train_df):,} records ({train_df['id_student'].nunique():,} unique students)")
print(f"Test partition:  {len(test_df):,} records ({test_df['id_student'].nunique():,} unique students)")
print(f"Train base rate: {train_df['withdrawn_after_cutoff'].mean():.2%}")
print(f"Test base rate:  {test_df['withdrawn_after_cutoff'].mean():.2%}")

In [ ]:
from student_retention.modeling.train import train_and_evaluate_all_models

# Execute cross-validation and baseline training
results = train_and_evaluate_all_models(clean_df, save_artifacts=True, random_state=42)

print("=== 5-Fold GroupKFold Cross-Validation Summary ===")
display(results["cv_summary"])

In [ ]:
print("=== Logistic Regression Odds Ratios (Top 10 Risk vs Protective Factors) ===")
lr_coefs = results["feature_importance_lr"]
display(lr_coefs.head(10))
display(lr_coefs.tail(10))

In [ ]:
print("=== Random Forest Gini Impurity Feature Importance (Top 10) ===")
display(results["feature_importance_rf"].head(10))